## Basic coding for AI systems

We work with AI models using text, but internally the models use what are called "tokens" to represent the basic atoms of processing (we'll focus on words). Let's see how we would use a large text base to create a tokenizer.  We'll use the collected works of Jane Austen. The tokenizer code is taken from the examples provided by Lightening AI.  
  
We create a class from which we can call the encode and decode functions, which turn words into tokens and tokens into words. At this step we're not using these functions, but we are preparing the dictionary _vocab_ with which they function. 

In [1]:
import re

class Tokenizer:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}
  def encode(self,text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self,ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text

with open("janes.txt","r",encoding="utf-8") as f:
  raw_text=f.read()

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item for item in preprocessed if item]
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab={token:integer for integer,token in enumerate(all_words)}
print("Vocabulary size: ",vocab_size)


Vocabulary size:  19151


## Tokenizing and recovering text

Now that we have a dictionary we can use it to turn a word into a token, and recover a word from a token.

In [ ]:
# ...following on
tokenizer=Tokenizer(vocab)

text="It is a truth universally acknowledged"
print("Token stream: ",ids)

ids=tokenizer.encode(text)
print("Recovery:     ",tokenizer.decode(ids))


## Creating the Model

Let's finish off our dive into the innards of an AI Generative model by looking at how we construct a model. To do this we'll use a python library called pytorch which contains the functions we need.  
We won't use our tokenizer, as we can also use a tokenizer library which provides high quality tokenization.  This is valled tiktoken.  
The first thing we do is to import the libraries we need. We'll also use a library from the litgpt folks which provides the transformation and normalisation functions we'll need.  

To start with, we import the libraries we'll be using and then define the configuration of the GPTModel we're building.  In this example, we're using a vocabulary of over 50K tokens and we're defining the embedding layer, normalisation layer, and multi-head layer sizes. We can ignore the drop rate and bias.  The context length is the amount of text our model can handle at any one time.


In [6]:
import torch, tiktoken
from transnorm import TransformerBlock, LayerNorm

GPT_CONFIG_124M = {
   "vocab_size": 50257,
   "context_length": 1024,
   "emb_dim": 768,
   "n_heads": 12,
   "n_layers": 12,
   "drop_rate": 0.0,
   "qkv_bias": False
}


## Setting up the model class

Next we create a class for the model which will create an instance of the model according to the configuration we provide, and includes the feed forward function. We won't go into the code in detail, suffice to observe that it sets up the two initial embedding sactions, creates the transformer blocks for each of the layers were using, and then sets up the final layer.  
The feed forward function produces what is known as logits, which are the raw, unnormalised predictions generated by the model before we apply the final layer transformation. In simple terms, they are the logarithm of the probabilities used to predict the answer for each possible answer.  
Finally, we create the model based on the configuration we previously defined.

In [19]:
# following on..
#
class GPTModel(torch.nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.tok_emb  = torch.nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb  = torch.nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = torch.nn.Dropout(cfg["drop_rate"])
    self.trf_blocks = torch.nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = torch.nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
  def forward(self,in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

model=GPTModel(GPT_CONFIG_124M)


## Generating and running the Model

Next we'll define our encoding using tiktoken, and we'll use the gpt2 byte-pair encoding scheme.  
The first thimng we do is set training mode and run a number of four word phrases in to train the model. Of course, a real world training would use billions of words, but this is just an example of the technique.  
We'll then switch to evaluation mode and provide the start of a sentence for the model to complete. We'll loop ten times generating a new word at each iteration by selecting the most probable entry in the solution vector. Every time we add a word, we use the updated text as the input to the next round.  
Finally we decode the output and display it. The result we get, of course, reflects the effort we put into training it ;-)

In [ ]:
#  following on...
#
tokenizer=tiktoken.get_encoding("gpt2")
model.train()
batch=[]
text="The comfortable home and"; batch.append(torch.tensor(tokenizer.encode(text)))
text="Each day some of"; batch.append(torch.tensor(tokenizer.encode(text)))
text="and had lived nearly"; batch.append(torch.tensor(tokenizer.encode(text)))
text="world with very little"; batch.append(torch.tensor(tokenizer.encode(text)))
text="To be or not"; batch.append(torch.tensor(tokenizer.encode(text)))
text="young for her age"; batch.append(torch.tensor(tokenizer.encode(text)))
text="father of at most"; batch.append(torch.tensor(tokenizer.encode(text)))
text="mother of the house"; batch.append(torch.tensor(tokenizer.encode(text)))
batch.append(torch.tensor(tokenizer.encode(text1)))
batch.append(torch.tensor(tokenizer.encode(text2)))
batch=torch.stack(batch,dim=0)
model(batch)

model.eval()
start_context = "The future of man is"
encoded = tokenizer.encode(start_context)
idx = torch.tensor(encoded).unsqueeze(0)
context_size=GPT_CONFIG_124M["context_length"]

for _ in range(10):
  idx_cond = idx[:, -context_size:]
  with torch.no_grad():
    logits = model(idx_cond)
    logits = logits[:, -1, :]
    probas = torch.softmax(logits, dim=1)
    idx_next = torch.argmax(probas, dim=-1, keepdim=True)
    idx = torch.cat((idx, idx_next), dim=1)

decoded_text = tokenizer.decode(idx.squeeze(0).tolist())
print("Untrained prediction: ",decoded_text)

